In [2]:
import pandas as pd
import yfinance as yf
import warnings
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns

# Suppress warnings
warnings.filterwarnings("ignore")

# List of 100 stock tickers
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'NFLX', 'BABA', 'V', 'JPM', 
           'JNJ', 'WMT', 'PG', 'DIS', 'MA', 'HD', 'UNH', 'PYPL', 'BAC', 'INTC', 'CMCSA']

# Download historical data for the stocks, suppress progress bar
try:
    data = yf.download(tickers, start="2020-01-01", end="2023-01-01", progress=False)['Adj Close']
except Exception as e:
    print(f"Error downloading data: {e}")
    data = pd.DataFrame()  # Initialize data as empty DataFrame if download fails

# Check if data is empty before proceeding
if not data.empty:
    # Calculate historical returns
    returns = data.pct_change().mean() * 252

    # Sort tickers based on returns
    sorted_tickers = returns.sort_values(ascending=False).index.tolist()

    # Function to group tickers into sets of 10
    def group_tickers(tickers, group_size=10):
        return [tickers[i:i + group_size] for i in range(0, len(tickers), group_size)]

    # Group the sorted tickers
    grouped_tickers = group_tickers(sorted_tickers)

    # Function to fetch stock data and calculate portfolio metrics
    def analyze_portfolio(tickers):
        try:
            # Download stock data
            data = yf.download(tickers, start="2020-01-01", end="2023-01-01", progress=False)
            prices = data['Adj Close']
            volumes = data['Volume']

            # Calculate expected returns and sample covariance matrix
            mu = expected_returns.mean_historical_return(prices)
            S = risk_models.sample_cov(prices)

            # Optimize for maximum Sharpe ratio
            ef = EfficientFrontier(mu, S)
            weights = ef.max_sharpe()
            cleaned_weights = ef.clean_weights()

            # Calculate the expected annual return, annual volatility, and Sharpe ratio
            performance = ef.portfolio_performance(verbose=False)  # Suppress print

            # Fetch additional stock data
            stock_info = yf.Tickers(tickers)

            stock_data = []
            for ticker in tickers:
                try:
                    info = stock_info.tickers[ticker].info
                    stock_data.append({
                        'Ticker': ticker,
                        'Price': prices[ticker].iloc[-1],
                        'Weight (%)': round(cleaned_weights[ticker] * 100, 2),  # Convert weight to percentage
                        'Volume': volumes[ticker].iloc[-1],
                        'Previous Volume': volumes[ticker].iloc[-2],
                        'All-Time High': info.get('fiftyTwoWeekHigh'),
                        '52-Week High': info.get('fiftyTwoWeekHigh'),
                        'Target': info.get('targetMeanPrice')
                    })
                except Exception:
                    continue  # Ignore errors silently
            return stock_data, performance

        except ValueError:
            return None, None

    # Analyze each group and display the results
    for i, group in enumerate(grouped_tickers):
        stock_data, performance = analyze_portfolio(group)
        if stock_data:
            df = pd.DataFrame(stock_data)
            print(f"\nSet {i + 1} Portfolio Performance:")
            print(f"Expected annual return: {performance[0]:.2%}")
            print(f"Annual volatility: {performance[1]:.2%}")
            print(f"Sharpe Ratio: {performance[2]:.2f}")
            print("\nStock Data:")
            print(df)
else:
    print("No data to analyze.")



Set 1 Portfolio Performance:
Expected annual return: 39.69%
Annual volatility: 39.42%
Sharpe Ratio: 0.96

Stock Data:
  Ticker       Price  Weight (%)     Volume  Previous Volume  All-Time High  \
0   TSLA  123.180000       41.02  157777300        221923300         271.00   
1   NVDA   14.604384        0.00  310490000        354923000         140.76   
2    UNH  516.237122       58.98    1849600          1379700         607.94   
3   AAPL  128.719360        0.00   77034200         75703700         237.23   
4   MSFT  236.420120        0.00   21938500         19770700         468.35   
5     HD  301.541016        0.00    2581600          1559100         401.11   
6  GOOGL   88.012070        0.00   23986300         23333500         191.75   
7     PG  145.065247        0.00    4532000          3809100         177.94   
8     MA  344.090363        0.00    1617600          1459800         501.80   
9    JNJ  167.519562        0.00    4216600          2828800         168.85   

   52-Week 

In [4]:
import pandas as pd
import yfinance as yf
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns

# List of 100 stock tickers
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'FB', 'NVDA', 'NFLX', 'BABA', 'V', 
           'JPM', 'JNJ', 'WMT', 'PG', 'DIS', 'MA', 'HD', 'UNH', 'PYPL', 'BAC', 
           'INTC', 'CMCSA', 'ADBE', 'PFE', 'KO', 'PEP', 'MRK', 'T', 'CSCO', 'XOM', 
           'VZ', 'ABT', 'CVX', 'NKE', 'LLY', 'MCD', 'MDT', 'CRM', 'HON', 'COST', 
           'AMGN', 'AVGO', 'TXN', 'QCOM', 'NEE', 'PM', 'UNP', 'LIN', 'ACN', 'MS', 
           'IBM', 'ORCL', 'TMO', 'SBUX', 'GS', 'BLK', 'RTX', 'SPGI', 'ISRG', 'NOW', 
           'CAT', 'MMM', 'GE', 'DHR', 'LMT', 'ADP', 'MDLZ', 'SYK', 'ZTS', 'GILD', 
           'AMT', 'CB', 'CI', 'SCHW', 'PLD', 'USB', 'C', 'TGT', 'MO', 'DUK', 
           'SO', 'BDX', 'ANTM', 'BKNG', 'CCI', 'APD', 'ICE', 'MMC', 'AON', 'FIS', 
           'FISV', 'EW', 'ITW', 'HUM', 'CL', 'EL', 'AEP', 'PSA', 'WM', 'D', 'ETN']

# Download historical data for the stocks
data = yf.download(tickers, start="2020-01-01", end="2023-01-01")['Adj Close']

# Calculate historical returns
returns = data.pct_change().mean() * 252

# Sort tickers based on returns
sorted_tickers = returns.sort_values(ascending=False).index.tolist()

# Function to group tickers into sets of 10
def group_tickers(tickers, group_size=10):
    return [tickers[i:i + group_size] for i in range(0, len(tickers), group_size)]

# Group the sorted tickers
grouped_tickers = group_tickers(sorted_tickers)

# Function to fetch stock data and calculate portfolio metrics
def analyze_portfolio(tickers):
    try:
        data = yf.download(tickers, start="2020-01-01", end="2023-01-01")
        prices = data['Adj Close']
        volumes = data['Volume']

        # Calculate expected returns and sample covariance matrix
        mu = expected_returns.mean_historical_return(prices)
        S = risk_models.sample_cov(prices)

        # Optimize for maximum Sharpe ratio
        ef = EfficientFrontier(mu, S)
        weights = ef.max_sharpe()
        cleaned_weights = ef.clean_weights()

        # Convert weights to percentages
        percentage_weights = {ticker: weight * 100 for ticker, weight in cleaned_weights.items()}

        # Calculate the expected annual return, annual volatility, and Sharpe ratio
        performance = ef.portfolio_performance(verbose=True)

        # Fetch additional stock data
        stock_info = yf.Tickers(tickers)

        stock_data = []
        for ticker in tickers:
            try:
                info = stock_info.tickers[ticker].info
                stock_data.append({
                    'Ticker': ticker,
                    'Price': prices[ticker].iloc[-1],
                    'Weight (%)': round(percentage_weights[ticker], 2),  # Display weight as percentage
                    'Volume': volumes[ticker].iloc[-1],
                    'Previous Volume': volumes[ticker].iloc[-2],
                    'All-Time High': info['fiftyTwoWeekHigh'],
                    '52-Week High': info['fiftyTwoWeekHigh'],
                    'Target': info['targetMeanPrice'] if 'targetMeanPrice' in info else None
                })
            except Exception as e:
                print(f"Skipping invalid/unavailable ticker {ticker}: {e}")
        return stock_data, performance

    except ValueError as e:
        print(f"Error processing tickers: {tickers}. Moving on... Error: {e}")
        return None, None

# Analyze each group and display the results in the notebook
for i, group in enumerate(grouped_tickers):
    stock_data, performance = analyze_portfolio(group)
    if stock_data:
        df = pd.DataFrame(stock_data)
        print(f"\nSet {i + 1} Portfolio Performance:")
        print(f"Expected annual return: {performance[0]:.2%}")
        print(f"Annual volatility: {performance[1]:.2%}")
        print(f"Sharpe Ratio: {performance[2]:.2f}")
        print("\nStock Data:")
        display(df)


[*********************100%***********************]  101 of 101 completed

3 Failed downloads:
['FB', 'ANTM']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
['FISV']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2020-01-01 -> 2023-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1577854800, endDate = 1672549200")')
[*********************100%***********************]  10 of 10 completed


Expected annual return: 46.3%
Annual volatility: 31.8%
Sharpe Ratio: 1.39

Set 1 Portfolio Performance:
Expected annual return: 46.25%
Annual volatility: 31.83%
Sharpe Ratio: 1.39

Stock Data:


,Ticker,Price,Weight (%),Volume,Previous Volume,All-Time High,52-Week High,Target
0,TSLA,123.180000,22.66,157777300,221923300,271.000,271.000,210.71
1,NVDA,14.604384,0.00,310490000,354923000,140.760,140.760,147.04
2,LLY,360.513763,72.28,1388000,1077900,972.530,972.530,1003.35
3,AVGO,54.086731,0.00,14376000,18176000,185.162,185.162,192.71
4,XOM,103.987717,5.06,11799600,10534000,123.750,123.750,130.93
5,SCHW,80.956322,0.00,5058400,3378500,79.490,79.490,72.90
6,MS,79.596794,0.00,4455600,3461700,109.110,109.110,104.41
7,CVX,167.260254,0.00,5005100,4142100,171.700,171.700,172.13
8,UNH,516.237061,0.00,1849600,1379700,607.940,607.940,623.61
9,AAPL,128.719345,0.00,77034200,75703700,237.230,237.230,240.58


[*********************100%***********************]  10 of 10 completed


Expected annual return: 19.5%
Annual volatility: 25.6%
Sharpe Ratio: 0.68

Set 2 Portfolio Performance:
Expected annual return: 19.49%
Annual volatility: 25.56%
Sharpe Ratio: 0.68

Stock Data:


...

Error processing tickers: ['EW', 'AMT', 'BDX', 'AMZN', 'USB', 'CRM', 'D', 'CMCSA', 'PYPL', 'T']. Moving on... Error: at least one of the assets must have an expected return exceeding the risk-free rate


[*********************100%***********************]  10 of 10 completed

2 Failed downloads:
['FB', 'ANTM']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')


Error processing tickers: ['C', 'MMM', 'MDT', 'VZ', 'DIS', 'FIS', 'BABA', 'INTC', 'ANTM', 'FB']. Moving on... Error: Eigenvalues did not converge


[*********************100%***********************]  1 of 1 completed

1 Failed download:
['FISV']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2020-01-01 -> 2023-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1577854800, endDate = 1672549200")')


Error processing tickers: ['FISV']. Moving on... Error: at least one of the assets must have an expected return exceeding the risk-free rate


In [6]:
# Original list of tickers with duplicates
tickers = [
    "SBFM", "FFIE", "MAXZ", "SMX", "VVOS", "DNA", "LAC", 
    "SOLO", "NVTS", "IQ", "CLSK", "EGL", "ACHR", "VSAT", 
    "AKCCF", "MARA", "LI", "SNOW", "BITF", "HUT", "NVAX", 
    "ESTC", "IEP", "BA", "CUB", "TLRY", "AI", "SLDP", 
    "TBLA", "NUVB", "ASTS", "AMD", "LUNR", "SAVA", "DVAX", 
    "CRWD", "SVRA", "AVAV", "MRCY", "GERN", "HII", "KBR", 
    "SAIC", "ERJ", "LHX", "KTOS", "RTX", "BAH", "TDY", 
    "RKLB", "MELI", "PSN", "IAG", "CACI", "LDOS", "LMT", 
    "GD", "HWM", "TDG", "NOC", "PLTR", "CW", "BNZI", "SEEL","VSME",

    "QXO", "PBLA", "MLYS", "INZY", "IGIC", "COOP", "EYPT", "VFS", "GTLS","RDDT",
    "FVRR", "AMN", "BNTX", "CROX", "SPCE", "CRSP", "CCCC", "ILMN", "LDOS", "BLDR",
    "CAR", "ALZN", "ALT", "IMMR", "BYND", "FIVN", "SG", "SYM", "LU", "NKLA",
    "ALB", "SMMT", "CVNA", "DDOG", "FIVE", "APPS", "TWLO", "VKTX", "TGTX", "CAVA",
    "SHOP", "W", "DOCU", "CELH", "HUT", "MRNA", "CDE", "DJT", "OKTA",
    "AI", "ARM", "DLTR", "ZM", "UPST", "SNOW", "RBLX", "HIMS", "SQ", "ROKU",
    "GILD", "PINS", "COIN", "AMC", "MSTR", "RKLB", "TDOC", "MRK", "CORZ", "BBIO",
    "LI", "TSM", "PYPL", "ETSY", "NVAX", "SMCI", "GERN", "CMG", "CFLT", "IREN",
    "RIOT", "PTON", "GME", "CLSK", "ASTS", "SNAP", "GOOG", "MU", "DG",
    "U", "RIVN", "AFRM", "MARA", "AMD", "DELL", "PFE", "SOFI", "PLTR", "AMZN",
    "LUNR", "NIO", "INTC", "NVDA", "SAVA", "SOXL", "RMBS", "SIMO",

    'GME','MVST', 'NEGG', 'MVIS', 'GEO', 'RKT', 'WEN', 'PLTR', 'RBLX', 'MNMD', 'SAVA', 
    'BCRX','TLRY','SMCI','FFIE', 'BNED','QXO','DNA', 'NKLA','PBI', 'BB', 'AMC', 'HOOD',
    'NOK', 'LOGC', 'CLNE','SOS','CRSR', 'KOSS', 'CLOV','GWS', 'WGS', 'ATER', 'LENZ', 
    'CRBP', 'SEDG', 'BOIL','IRBT','LAZR', 'LUMN', 'PLUG','PACB','VINO', 'MGNX', 'SLRN', 
    'TTEC', 'EGIO', 'CABA','PLCE', 'WOLF','AGL','RTC', 'BKKT','GWH','FLYX', 'AZUL','SOXL',
    'NVAX', 'CCCC', 'MLYS', 'ALT','TGTX', 'RXRX', 'NIO','SMMT','ASTS', 'UPST', 'CAVA', 
    'CVNA', 'RIVN','AFRM', 'U', 'PATH','RKLB','MARA','HUT','CLSK', 'RIOT', 'CORZ', 'IREN',
    'WGMI', 'SOXL',
    
    "CHWY", "ASTS", "SNAP", "VKTX", "SMMT", "SOXL", "NIO", "OPEN", "SOXS", "IMRX",
    "SQQQ", "WBD", "SMX", "PLTR", "INTC", "MARA", "PLUG", "LCID", "U", "DJT",
    "FRGT", "SPI", "RIVN", "CCL", "CAVA", "CVNA", "NYCB", "WULF", "LIFT", "X",
    "TLRY", "RDFN", "WBA", "GEVO", "LUMN", "CLSK", "EGIO", "RIOT", "CDE", "CLF",
    "APLD", "RKLB", "AFRM", "PTON", "WOLF", "NVAX", "MRNA", "HOOD", "PATH", "PLCE",
    "LUNR", "FUBO", "PACB", "CAN", "TELL", "TNON", "EWZ", "ZI", "HL", "QH",
    "FTCI", "RUN", "RVNC", "WOOF", "CPNG", "IOVA", "GDXJ", "PYPL", "HBI", "W",
    "PENN", "APPS", "ROKU", "AGQ", "SAVA", "FIVE", "QXO", "VXRT", "LODE", "RLAY",
    "TVTX", "MNKD", "DVAX", "DRIP", "SPCE", "VRDN", "DOCU", "DOCS", "UPST", "Z",
    "ARKK", "TMF", "SHOP", "CNEY", "ALT", "RBLX", "SHLS", "NOVA", "UNG", "SG",
    "UA", "SE", "CCCC", "TGTX", "RXRX", "RDDT"
]

# Remove duplicates by converting to a set and back to a list
tickers = list(set(tickers))

# Output the length of the list after removing duplicates
print(len(tickers))


258


In [7]:
import pandas as pd
import yfinance as yf
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns

# List of 100 stock tickers
#tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'FB', 'NVDA', 'NFLX', 'BABA', 'V']

# Download historical data for the stocks
data = yf.download(tickers, start="2020-01-01", end="2023-01-01")['Adj Close']

# Calculate historical returns
returns = data.pct_change().mean() * 252

# Sort tickers based on returns
sorted_tickers = returns.sort_values(ascending=False).index.tolist()

# Function to group tickers into sets of 10
def group_tickers(tickers, group_size=10):
    return [tickers[i:i + group_size] for i in range(0, len(tickers), group_size)]

# Group the sorted tickers
grouped_tickers = group_tickers(sorted_tickers)

# Function to fetch stock data and calculate portfolio metrics
def analyze_portfolio(tickers):
    try:
        data = yf.download(tickers, start="2020-01-01", end="2023-01-01")
        prices = data['Adj Close']
        volumes = data['Volume']

        # Calculate expected returns and sample covariance matrix
        mu = expected_returns.mean_historical_return(prices)
        S = risk_models.sample_cov(prices)

        # Optimize for maximum Sharpe ratio
        ef = EfficientFrontier(mu, S)
        weights = ef.max_sharpe()
        cleaned_weights = ef.clean_weights()

        # Convert weights to percentages
        percentage_weights = {ticker: weight * 100 for ticker, weight in cleaned_weights.items()}

        # Calculate the expected annual return, annual volatility, and Sharpe ratio
        performance = ef.portfolio_performance(verbose=True)

        # Fetch additional stock data
        stock_info = yf.Tickers(tickers)

        stock_data = []
        for ticker in tickers:
            try:
                info = stock_info.tickers[ticker].info
                stock_data.append({
                    'Ticker': ticker,
                    'Price': prices[ticker].iloc[-1],
                    'Weight (%)': round(percentage_weights[ticker], 2),  # Display weight as percentage
                    'Volume': volumes[ticker].iloc[-1],
                    'Previous Volume': volumes[ticker].iloc[-2],
                    'All-Time High': info['fiftyTwoWeekHigh'],
                    '52-Week High': info['fiftyTwoWeekHigh'],
                    'Target': info['targetMeanPrice'] if 'targetMeanPrice' in info else None
                })
            except Exception as e:
                print(f"Skipping invalid/unavailable ticker {ticker}: {e}")
        return stock_data, performance

    except ValueError as e:
        print(f"Error processing tickers: {tickers}. Moving on... Error: {e}")
        return None, None

# Analyze each group and display the results in the notebook
for i, group in enumerate(grouped_tickers):
    stock_data, performance = analyze_portfolio(group)
    if stock_data:
        df = pd.DataFrame(stock_data)
        print(f"\nSet {i + 1} Portfolio Performance:")
        print(f"Expected annual return: {performance[0]:.2%}")
        print(f"Annual volatility: {performance[1]:.2%}")
        print(f"Sharpe Ratio: {performance[2]:.2f}")
        print("\nStock Data:")
        display(df)


[*********************100%***********************]  258 of 258 completed

16 Failed downloads:
['GWS', 'MAXZ', 'EGL']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2020-01-01 -> 2023-01-01)')
['LAC', 'ARM', 'SLRN', 'VFS', 'FLYX', 'CUB', 'MLYS', 'CORZ', 'RDDT', 'CAVA', 'VSME']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2020-01-01 -> 2023-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1577854800, endDate = 1672549200")')
['SOLO', 'LIFT']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
[*********************100%***********************]  10 of 10 completed


Expected annual return: 86.5%
Annual volatility: 106.7%
Sharpe Ratio: 0.79

Set 1 Portfolio Performance:
Expected annual return: 86.48%
Annual volatility: 106.73%
Sharpe Ratio: 0.79

Stock Data:


,Ticker,Price,Weight (%),Volume,Previous Volume,All-Time High,52-Week High,Target
0,SPI,0.860000,0.00,228800.0,249300.0,1.26,1.26,NaN
1,DJT,15.000000,3.43,538400.0,709700.0,79.38,79.38,NaN
2,KOSS,4.950000,0.00,29000.0,48100.0,18.73,18.73,NaN
3,GME,18.459999,48.38,2669500.0,3442800.0,64.83,64.83,7.88
4,SAVA,29.540001,15.85,1025800.0,1168600.0,42.20,42.20,103.00
5,MNMD,2.200000,4.86,961900.0,948100.0,12.22,12.22,26.25
6,VXRT,0.960000,9.11,3783200.0,4170400.0,1.54,1.54,4.83
7,MARA,3.420000,11.50,9615000.0,11226200.0,34.09,34.09,21.44
8,MVIS,2.350000,6.87,1754700.0,1936700.0,2.98,2.98,NaN
9,AMC,40.700001,0.00,1845080.0,2122530.0,11.88,11.88,4.64


[*********************100%***********************]  10 of 10 completed


Expected annual return: 142.7%
Annual volatility: 59.6%
Sharpe Ratio: 2.36

Set 2 Portfolio Performance:
Expected annual return: 142.69%
Annual volatility: 59.65%
Sharpe Ratio: 2.36

Stock Data:


...

[*********************100%***********************]  10 of 10 completed


Expected annual return: 3.7%
Annual volatility: 32.6%
Sharpe Ratio: 0.05

Set 15 Portfolio Performance:
Expected annual return: 3.67%
Annual volatility: 32.58%
Sharpe Ratio: 0.05

Stock Data:


,Ticker,Price,Weight (%),Volume,Previous Volume,All-Time High,52-Week High,Target
0,LDOS,102.741737,100.0,392200,423400,160.32,160.32,169.36
1,MU,49.544937,0.0,11989800,13204300,157.54,157.54,155.01
2,WULF,0.670000,0.0,502700,296100,6.51,6.51,6.56
3,CRSP,40.650002,0.0,1147200,1361300,91.10,91.10,81.21
4,BB,3.260000,0.0,7189400,7616400,4.90,4.90,4.69
5,LAZR,4.950000,0.0,5716300,5753900,4.62,4.62,2.59
6,ASTS,4.820000,0.0,2616700,2474000,39.08,39.08,37.98
7,LHX,200.036255,0.0,716600,898200,245.60,245.60,252.30
8,HIMS,6.410000,0.0,1630200,924500,25.74,25.74,21.93
9,CDE,3.360000,0.0,5041300,4486200,7.72,7.72,7.56


[*********************100%***********************]  10 of 10 completed


SolverError: Solver 'OSQP' failed. Try another solver, or solve with verbose=True for more information.

In [10]:

tickers=['MYI', 'FWRG', 'IBTE', 'HIGH', 'ACWI', 'KWEB', 'LCCN', 'MINT', 'SMTC', 'ERNA', 'CIBR', 'TRIP', 'SPXL', 'INTL', 'BOTZ', 'HDRO', 'ROOF', 'BEMO', 'AMAT', 'MMIT', 'AERS', 'GRID', 'MAGG', 'GHYG', 'ALUM', 'FDN', 'QCLN', 'HSTC', 'BLOK', 'IGC', 'BERI', 'COCO', 'CYN', 'BPCR', 'TEM', 'PIN', 'WHR', 'WOOD', 'BTEK', 'COMM', 'INFR', 'VERX', 'IBTL', 'MINV', 'SPGP', 'WLDS', 'IBTM', 'FUSI', 'ISPY', 'ESGB', 'MIDD', 'III', 'GROW', 'SONG', 'SEED', 'FXC', 'IBTG', 'SHYG', 'HSPX', 'EUDV', 'CRS', 'PSH', 'ATST', 'MRC', 'RCP', 'PCT', 'HRI', 'EOT', 'SOI', 'FSV', 'BBH', 'SDP', 'EWI', 'PHI', 'API', 'SOHO', 'BGS', 'HFEL', 'FAS', 'DIG', 'AEI', 'PAC', 'DIVI', 'UEM', 'INOV', 'IAT', 'EAT', 'ARR', 'ATS', 'ATR', 'USA', 'SJG', 'SST', 'GLDD', 'STKS', 'PYCR', 'EYE', 'STER', 'AVTX', 'EAF', 'EVI', 'GNW', 'EBC', 'AVTE', 'AAGIY', 'PKOH', 'AUB', 'AMSC', 'FDMT', 'FNLC', 'PFIS', 'OB', 'PHVS', 'OXM', 'PDSB', 'PDS', 'IMUX', 'PNGAY', 'ARRY', 'RCUS', 'KRNL', 'HSII', 'RPAY', 'ACCD', 'BRY', 'BASE', 'PLBY', 'ORGO', 'CATC', 'MPX', 'VHAQ', 'PAX', 'INNV', 'HBIO', 'DGICA', 'RAPT', 'DNN', 'EBTC', 'CRVL', 'SRRK', 'RYAN', 'HCKT', 'SCWX', 'XBIT', 'MRUS', 'CHX', 'CATY', 'NUVB', 'CUE', 'CSIQ', 'KAOOY', 'ATEX', 'UTMD', 'REPX', 'TSHA', 'PTGX', 'AMRK', 'VFF', 'MCBS', 'ATHA', 'APG', 'GDYN', 'MMYT', 'IPSC', 'MSBI', 'OPT', 'QURE', 'DAWN', 'DMRC', 'ODC', 'PCB', 'BRBS', 'RBB', 'SAR', 'BLFY', 'BH', 'DCBO', 'SPRO', 'FREE', 'REVG', 'HVT', 'IKNA', 'MEOH', 'GRTS', 'VICR', 'BCAB', 'ESCA', 'DIBS', 'BOLT', 'BV', 'SMMT', 'MGIC', 'ZVRA', 'CGBD', 'PMVP', 'DCOM', 'ATOM', 'SGRY', 'CABA', 'CRBU', 'NBTX', 'SFST', 'CSR', 'SKYT', 'OI', 'NRIX', 'MKFG', 'AGYS', 'JYNT', 'PLSE', 'CLSD', 'RLMD', 'ARQT', 'ALT', 'EVC', 'LEGH', 'NVEC', 'BHR', 'VINP', 'FRBA', 'LND', 'NWPX', 'SILV', 'KAI', 'KE', 'THFF', 'VTNR', 'MYE', 'ALDX', 'NOA', 'GHC', 'CLFD', 'DRIO', 'OSPN', 'NODK', 'ORRF', 'PKBK', 'TIGO', 'ACHL', 'BYDDY', 'FLXS', 'JLL', 'BYRN', 'CVM', 'AGMH', 'QNCX', 'CECO', 'ING', 'AMAL', 'WINA', 'TUYA', 'ACGL', 'FTHM', 'ALTR', 'FFIC', 'GAMC', 'ADV', 'ONEW', 'CIA', 'GDEN', 'HRZN', 'NATH', 'ULH', 'TKNO', 'AJX', 'FPH', 'HWKN', 'MEG', 'RCKY', 'ADAG', 'KYMR', 'CVGI', 'SCM', 'SA', 'ASA', 'PKE', 'LBRDA', 'VEL', 'LQDT', 'VSEC', 'PGC', 'FSUGY', 'ACET', 'PASG', 'MOLN', 'APLT', 'ADNT', 'CKHUY', 'HWC', 'TARS', 'BSRR', 'LOOP', 'MRSN', 'BGXX', 'MBIN', 'STEP', 'FMAO', 'WSC', 'RVP', 'CTRN', 'VET', 'USLM', 'TBBK', 'YORW', 'KOPN', 'NXE', 'RPID', 'TPVG', 'VOXX', 'BLUA', 'CZNC', 'MIRM', 'WVE', 'ZVIA', 'MX', 'IBOC', 'HITI', 'ETON', 'EBR', 'TTSH', 'RSVR', 'NKTX', 'MTA', 'ABCL', 'PRTS', 'NL', 'CIVB', 'CHK', 'AUTL', 'SWKH', 'TCMD', 'KMTUY', 'CLAR', 'ARKO', 'NIC', 'MCRI', 'PARR', 'KELYB', 'SFTBY', 'AMK', 'RLX', 'FRST', 'BCOV', 'CDXC', 'GWRS', 'UWMC', 'SPFI', 'PBH', 'BRDG', 'BY', 'KURA', 'OPRX', 'ETNB', 'HTBI', 'GHLD', 'GTHX', 'HTH', 'MEC', 'KMDA', 'BIVI', 'ONVO', 'STN', 'CNM', 'SMTI', 'HBB', 'ALTG', 'GCMG', 'FDUS', 'CRC', 'APYX', 'RELY', 'CARE', 'NVGS', 'LNN', 'AVAL', 'AROW', 'URGN', 'FCBC', 'GATO', 'RBBN', 'HSTM', 'CTSO', 'CMTL', 'PRTG', 'CLNN', 'INVE', 'CALT', 'COGT', 'ARCT', 'CDNA', 'WTBA', 'INVZ', 'ABUS', 'LCUT', 'LAW', 'AVNW', 'IR', 'CNXN', 'AMR', 'CVEO', 'TAC', 'CLBK', 'BEEM', 'RFL', 'SBOW', 'ATLC', 'REKR', 'LUNA', 'DSP', 'IBEX', 'HOV', 'TG', 'TRIN', 'UONEK', 'CCBG', 'REAX', 'NOTV', 'MORF', 'KELYA', 'ACEL', 'TRC', 'MITK', 'PLPC', 'HIFS', 'CNTY', 'GPAC', 'AVIR', 'RXRX', 'LUMN', 'GLSI', 'BIOX', 'ORMP', 'MPB', 'INSE', 'ALLK', 'SIBN', 'FHTX', 'BCBP', 'HFWA', 'AVD', 'MSEX', 'MLR', 'RVMD', 'EEX', 'IGIC', 'SGC', 'BELFB', 'VERA', 'WHF', 'BMRC', 'UBS', 'RICK', 'ANNX', 'AGS', 'NEON', 'APEI', 'GRC', 'SOPH', 'BEKE', 'XYF', 'LL', 'PPTA', 'ITI', 'FYBR', 'PCVX', 'HBCP', 'TMCI', 'ZIM', 'VAL', 'BDSX', 'OSBC', 'NEGG', 'OFIX', 'DSNKY', 'INFU', 'CMBM', 'OSW', 'CRMD', 'SLI', 'ABOS', 'SNSE', 'EPIX', 'COOP', 'ARES', 'NTTYY', 'SANA', 'DSX', 'FMBH', 'PEBO', 'TCBK', 'IMOS', 'TPB', 'MNRO', 'UTL', 'LFMD', 'NAUT', 'DY', 'MOFG', 'OLO', 'NMG', 'RMNI', 'PRTH', 'CPRX', 'STRO', 'SRCE', 'HONE', 'SMPL', 'PLRX', 'RNA', 'PLYA', 'EXK', 'SHYF', 'CRAI', 'BRLT', 'BSET', 'BMEA', 'KDDIY', 'ALVR', 'LDI', 'MGTX', 'MRNS', 'KIRK', 'KIDS', 'JWSM', 'INMB', 'APLS', 'QMCO', 'TITN', 'GHRS', 'UPST', 'SPRB', 'TIXT', 'ACNB', 'ANAB', 'VRCA', 'FSBC', 'RMR', 'IRMD', 'VWE', 'LEU', 'ALGS', 'QDEL', 'CURI', 'IPI', 'ZYME', 'IMAB', 'VERU', 'PWP', 'INST', 'MTX', 'RBCAA', 'CLSK', 'THR', 'QSI', 'SPT', 'FMNB', 'ATRC', 'ZOM', 'PARAA', 'GNTY', 'IESC', 'OLK', 'METC', 'CAMT', 'BMTX', 'CRDF', 'ZENV', 'WMS', 'ROAD', 'THRY', 'COHU', 'PTSI', 'IFS', 'FORA', 'BROG', 'VUZI', 'BWB', 'ARTNA', 'KNTK', 'LSEA', 'ASPN', 'ETWO', 'FULC', 'UFPT', 'IGMS', 'BVS', 'NEWT', 'XENE', 'WOW', 'MNTK', 'FDBC', 'SNCY', 'PHAT', 'BALY', 'CWAN', 'STIM', 'PROF', 'CPAC', 'FOA', 'ALRS', 'GP', 'BHB', 'GPOR', 'HURN', 'PFC', 'NPCE', 'FLNG', 'EWTX', 'BBW', 'SRDX', 'GSL', 'AON', 'CATO', 'AVXL', 'KRUS', 'MNSO', 'OFLX', 'POWW', 'PACK', 'PBFS', 'CASS', 'SAVA', 'VRA', 'XOMA', 'HTBK', 'DZSI', 'VRNA', 'INBK', 'ABSI', 'NSSC', 'SLGL', 'ALXO', 'HLNE', 'PSEC', 'ITRN', 'MGY', 'LXFR', 'TRNS', 'CVLT', 'RDNT', 'NBN', 'CFB', 'AKYA', 'NGVC', 'ERO', 'IMTX', 'OPY', 'SWTX', 'WAFD', 'QSG', 'AMSWA', 'PVBC', 'IMCR', 'KARO', 'AMTX', 'SJW', 'AMRC', 'CVRX', 'MYTE', 'PRQR', 'CTMX', 'QNST', 'BBCP', 'RDVT', 'PCYO', 'VMD', 'GLUE', 'OPRT', 'OCUL', 'KB', 'ANVS', 'CELC', 'CGAU', 'CNTA', 'EVLV', 'LBRT', 'TGLS', 'MBWM', 'MDXG', 'AXTI', 'VERV', 'TFII', 'FOUR', 'ONDS', 'BFST', 'NDLS', 'MWA', 'OLN', 'BXC', 'IBCP', 'ASLE', 'SERA', 'UUUU', 'ASTS', 'LVTX', 'HRTG', 'GEF', 'STXS', 'VOR', 'BRSP', 'CBZ', 'OPCH', 'STKL', 'PECO', 'DFIN', 'OLMA', 'ATEC', 'SHC', 'TSEM', 'CDZI', 'BYSI', 'IDYA', 'CBNK', 'TIMB', 'PERI', 'CDMO', 'VALU', 'TIPT', 'PSTX', 'MESA', 'WSBF', 'HFFG', 'NATR', 'PSNL', 'LSF', 'AXNX', 'AGO', 'VTEX', 'CMRX', 'BFC', 'SENEA', 'PSN', 'KALV', 'CMMB', 'GNK', 'VLGEA', 'SPRY', 'KZR', 'CNTB', 'FOX', 'EYPT', 'LUNG', 'BLBD', 'CRNX', 'OOMA', 'KGC', 'IMVT', 'DAC', 'STRL', 'MCB', 'MAPS', 'NRIM', 'ICAD', 'CCB', 'CCNE', 'HPK', 'GBIO', 'CCAP', 'BCYC', 'LMNR', 'KROS', 'HBT', 'UONE', 'AOMR', 'CTOS', 'LASR', 'ASXC', 'RCI', 'ALCO', 'HQI', 'CVLG', 'ZNTL', 'WILC', 'TERN', 'PRPL', 'NOG', 'SII', 'MCBC', 'VPG', 'CTKB', 'CERE', 'HOWL', 'FLL', 'LIND', 'ADCT', 'RLGT', 'FORTY', 'CLPT', 'SKIN', 'TMDX', 'CLPR', 'MVBF', 'AMBA', 'SMBC', 'NRC', 'IMXI', 'UTZ', 'KRT', 'ELLO', 'ITIC', 'GSBC', 'FC', 'IDT', 'SLAM', 'MDGL', 'FORR', 'IVA', 'JANX', 'MSC', 'CIX', 'ACTG', 'CHRD', 'VIRX', 'ESTA', 'GABC', 'SILK', 'EVSb', 'SHLd', 'PSMd', 'INHd', 'WCHd', 'MLPd', 'RENEu', 'TRAC', 'AOM', 'MOON', 'EDV', 'BCG', 'DOCS', 'SPR', 'ADVT', 'FLO', 'CHH', 'PEG', 'SNX', 'TMO', 'EOG', 'CLX', 'NET', 'STM', 'TEAM', 'TPX', 'VNET', 'AEO', 'INSG', 'EML', 'KEYS', 'GAL', 'DIS', 'GENI', 'SUP', 'PEN', 'NBB', 'MTC', 'VINO', 'THS', 'TRU', 'ECOR', 'CIZ', 'ALK', 'KARs', 'VERv', 'STRv', 'RRBI', 'IDCC', 'TRN', 'CNA', 'CCL', 'CRH', 'FERG', 'RTC', 'CARD', 'MTW', 'CTEC', 'ELCO', 'TRT', 'LPA', 'MAC', 'SSTY', 'PRIM', 'ARBB', 'COM', 'JAN', 'DKL', 'TRAK', 'MIND', 'GOOD', 'MIN', 'CNS', 'SPEC', 'IGE', 'IRON', 'EDEN', 'THG', 'HAS', 'JUST', 'IPO', 'FLTR', 'LRE', 'PTEC', 'WPP', 'SMIN', 'RIO', 'AZN', 'KNOS', 'ESNT', 'GRI', 'MGAM', 'BBY', 'GNS', 'CWK', 'IHG', 'RKT', 'RDW', 'PRU', 'ADM', 'SVT', 'RTO', 'PAG', 'AHT', 'FRES', 'SGE', 'NXT', 'AAL', 'DFS', 'ARC', 'FDP', 'POLR', 'HCM', 'IPX', 'GEN', 'SPI', 'PLUS', 'SEE', 'PPH', 'YOU', 'CRW', 'APH', 'AMS', 'ITM', 'CAML', 'FAN', 'PRTC', 'STEM', 'CNC', 'AEP', 'BMY', 'BOWL', 'BOOT', 'LIT', 'FTC', 'PXS', 'BMN', 'CYAN', 'XTR', 'STX', 'SYM', 'TIME', 'KOD', 'IGR', 'SAA', 'AURA', 'EME', 'RRR', 'STAF', 'FAB', 'OPTI', 'XPP', 'ECO', 'ACP', 'BAR', 'OVB', 'PEB', 'KEFI', 'AGL', 'BIRD', 'MOS', 'BOIL', 'HUM', 'FDEV', 'CMCL', 'SUN', 'MERC', 'UFO', 'CRDL', 'ASC', 'FSTA', 'PHAR', 'PAY', 'SDY', 'DIA', 'TRI', 'COST', 'OMI', 'UTG', 'DLN', 'BLND', 'LAND', 'AGR', 'SAFE', 'CAL', 'BIIB', 'UA', 'SEDG', 'USFD', 'EC', 'PLNT', 'DT', 'POWL', 'ZI', 'MAIN', 'WEX', 'M', 'NXPI', 'CARS', 'FSLR', 'ALV', 'GTLS', 'XYL', 'SPB', 'SEIC', 'TLK', 'KEP', 'OR', 'PD', 'WRB', 'TPC', 'WEC', 'TMP', 'BLMN', 'SCI', 'RXT', 'SKT', 'DXCM', 'CRNC', 'LOPE', 'IFF', 'AVAV', 'BFAM', 'MS', 'RIG', 'GOTU', 'BSAC', 'UCBI', 'NMIH', 'NFBK', 'AWI', 'CARG', 'HCSG', 'RGNX', 'UMC', 'DG', 'ICLR', 'FTI', 'VRE', 'VVX', 'MTZ', 'HUN', 'GWRE', 'BCO', 'ONB', 'TME', 'CNX', 'TS', 'PFLT', 'SIMO', 'REG', 'EXPO', 'AKTS', 'AEIS', 'PTEN', 'NOAH', 'WH', 'AVGO', 'PODD', 'XGN', 'ACA', 'VIST', 'APAM', 'TGI', 'NOVT', 'MLKN', 'MRK', 'GBX', 'AAT', 'ULTA', 'FI', 'DXC', 'TTEK', 'CRON', 'TMUS', 'MU', 'VREX', 'CHGG', 'NR', 'MC', 'MCD', 'PFE', 'LUV', 'LFUS', 'FFBC', 'GLRE', 'HBAN', 'SNA', 'SF', 'MYRG', 'TECK', 'FCF', 'CYTK', 'CXW', 'DOOO', 'TPR', 'EPC', 'KMB', 'PBPB', 'ICFI', 'DKS', 'FSS', 'SAGE', 'NARI', 'MMM', 'GFL', 'SQ', 'PRK', 'IH', 'CSV', 'IEX', 'LOGI', 'GCI', 'LII', 'NVCR', 'TSN', 'SPWR', 'FOXF', 'COP', 'LX', 'VFC', 'SWBI', 'INTC', 'CSCO', 'ACHC', 'BIO', 'KRNT', 'MTDR', 'RUN', 'AHCO', 'WRK', 'TENB', 'PCTY', 'RRC', 'MMC', 'DDD', 'ASTE', 'LBRDK', 'TGT', 'AR', 'BJRI', 'SEER', 'VTOL', 'TELL', 'ENR', 'BTBT', 'CALX', 'BWXT', 'WTFC', 'IPGP', 'IX', 'GCO', 'CLNE', 'LHX', 'GBCI', 'PII', 'SWK', 'HMN', 'NOW', 'WRAP', 'STOK', 'ESS', 'MTLS', 'JEF', 'LNTH', 'XEL', 'ARMK', 'PAYX', 'AKRO', 'ARVN', 'IQ', 'HAYN', 'SLP', 'DGII', 'ALLE', 'GATX', 'RMBS', 'SHG', 'MO', 'MTG', 'GFI', 'WRLD', 'MDT', 'CBOE', 'BECN', 'TNET', 'MHO', 'FBP', 'CAH', 'OTTR', 'PBR', 'ARCH', 'RH', 'GNRC', 'GAN', 'CALM', 'BOKF', 'JNPR', 'SE', 'MCHP', 'SWN', 'PRDO', 'GPI', 'CDW', 'VSH', 'NMR', 'GIB', 'MTRX', 'DJCO', 'FBNC', 'NEU', 'CRSP', 'FMX', 'RYI', 'THRM', 'LEN', 'GWW', 'UNM', 'FUL', 'PINS', 'MTSI', 'GRVY', 'CZR', 'TBI', 'XNET', 'PBA', 'ALGT', 'OMCL', 'TEL', 'SENS', 'DOW', 'LGND', 'RDY', 'YEXT', 'RPRX', 'EW', 'HRTX', 'SXT', 'FRME', 'IRDM', 'AER', 'STT', 'NTAP', 'SSSS', 'ED', 'WIMI', 'AROC', 'HRL', 'MCY', 'WCN', 'VSTO', 'HUYA', 'TWLO', 'A', 'BANF', 'NTLA', 'MTRN', 'DLX', 'URI', 'ACAD', 'QS', 'ROST', 'CCJ', 'WD', 'WWW', 'AXS', 'TCS', 'VAC', 'FINV', 'CW', 'SM', 'MAS', 'KOP', 'ATEN', 'SBSW', 'ANIX', 'FORM', 'INFN', 'CHE', 'MRO', 'LOMA', 'SNDR', 'SON', 'OMF', 'SR', 'SHOP', 'TYL', 'NEO', 'EFX', 'CL', 'JACK', 'GRWG', 'SHAK', 'UMBF', 'SQM', 'GKOS', 'MGEE', 'AXTA', 'CRWD', 'AZUL', 'DLR', 'PAM', 'TTC', 'HLT', 'OUT', 'STAA', 'SNDX', 'CPT', 'NAVI', 'HPQ', 'CCEP', 'XNCR', 'ANSS', 'WMG', 'WLY', 'TAL', 'CAT', 'CEVA', 'HRB', 'PK', 'WEN', 'NBHC', 'SMAR', 'TREE', 'W', 'NRG', 'HEAR', 'REX', 'VECO', 'ITUB', 'WWD', 'TCPC', 'ROG', 'ALLO', 'RHP', 'ARE', 'IBTX', 'GLOB', 'CEIX', 'TOUR', 'RDFN', 'EDIT', 'FHN', 'BBIO', 'SSRM', 'MOV', 'RLJ', 'SPTN', 'YPF', 'XRAY', 'BR', 'RF', 'JMIA', 'HIBB', 'SRI', 'LNT', 'OCFC', 'DXPE', 'RGP', 'SPG', 'RLI', 'GT', 'MATW', 'HSIC', 'ETN', 'JAZZ', 'AMED', 'WERN', 'SAIA', 'RHI', 'APD', 'HBNC', 'SUM', 'TDY', 'KEY', 'OHI', 'DELL', 'LSTR', 'ANF', 'CRUS', 'JBL', 'CPK', 'QCRH', 'CME', 'VIAV', 'BB', 'GSAT', 'PLD', 'MD', 'RM', 'HIG', 'DOV', 'FDX', 'BC', 'CNSL', 'SPGI', 'JBT', 'LAD', 'KDP', 'NVT', 'AUPH', 'PHM', 'BDC', 'PTON', 'WYNN', 'CAE', 'SWKS', 'HLF', 'PNC', 'VCYT', 'LMT', 'ACMR', 'WIRE', 'LNG', 'ADI', 'GHG', 'WMB', 'RILY', 'MAN', 'OZK', 'BLKB', 'ALLT', 'BEPC', 'BPT', 'AVO', 'LITE', 'PFG', 'ASPS', 'NBTB', 'SBH', 'DIN', 'PBF', 'CTRE', 'ALL', 'SCCO', 'WAL', 'HRMY', 'HE', 'NVR', 'GPRE', 'HXL', 'FCEL', 'ETRN', 'UIS', 'DNMR', 'SFIX', 'BKH', 'ZION', 'MKTX', 'SWX', 'AXON', 'PINC', 'ORLY', 'UNH', 'NUE', 'IOVA', 'WGO', 'SITC', 'ESRT', 'HGV', 'KLIC', 'LTRN', 'COLB', 'MAA', 'SNOW', 'ZBH', 'KEX', 'RSI', 'MDU', 'BILL', 'GBDC', 'FUTU', 'CASY', 'GRBK', 'CHUY', 'CHWY', 'PRTA', 'PAHC', 'HYFM', 'MGNI', 'CMCO', 'ATKR', 'PNM', 'CVNA', 'LANC', 'ANDE', 'CORT', 'UNP', 'TX', 'BCPC', 'VHI', 'INVH', 'PSTG', 'PFGC', 'ENV', 'MRTN', 'WLDN', 'ARW', 'OSK', 'NFE', 'WTRG', 'PCRX', 'BUSE', 'FELE', 'HELE', 'MLM', 'BKR', 'PRO', 'LIVN', 'DOX', 'WFC', 'GFF', 'HQY', 'UAA', 'TNDM', 'PAR', 'PLMR', 'ESPR', 'USB', 'YUM', 'CWST', 'CNMD', 'DLB', 'PHG', 'CHTR', 'CIEN', 'CSX', 'OTEX', 'PM', 'IDA', 'COLD', 'STNE', 'MXL', 'BNS', 'ZEUS', 'AMH', 'AEE', 'LI', 'HTHT', 'PRA', 'ENIC', 'MBUU', 'FFIV', 'DNLI', 'MSCI', 'WELL', 'B', 'IMKTA', 'PH', 'BAK', 'MAG', 'CWH', 'KOF', 'ALB', 'XPRO', 'EA', 'NTRA', 'REGN', 'GO', 'DENN', 'GPC', 'QUAD', 'NI', 'UTHR', 'WSFS', 'AAP', 'SMFG', 'BCC', 'HEES', 'VRSN', 'TTGT', 'EHC', 'SIRI', 'PNFP', 'HIMX', 'ROL', 'KN', 'CTS', 'POWI', 'BTG', 'VSTA', 'HAIN', 'EEFT', 'TDS', 'NGVT', 'WB', 'WPM', 'FISI', 'NVS', 'EVBG', 'CVCO', 'AMPH', 'SSD', 'AZO', 'USNA', 'VNT', 'TPIC', 'UBER', 'SO', 'SXI', 'AXL', 'SCSC', 'PTCT', 'PLOW', 'DOCU', 'ABBV', 'SSNC', 'AGX', 'EGHT', 'INSW', 'CAG', 'ARWR', 'BOX', 'MOD', 'ADAP', 'NBR', 'VKTX', 'ORCL', 'MVIS', 'CAC', 'VNDA', 'MRNA', 'ICHR', 'T', 'LSCC', 'SLQT', 'UDR', 'CYH', 'OSUR', 'BIPC', 'ORI', 'VRNT', 'MBI', 'KPTI', 'EXPI', 'ETR', 'GDS', 'CNNE', 'FATE', 'OVV', 'PHR', 'VERI', 'FHB', 'TEO', 'NVMI', 'VXRT', 'WIT', 'TD', 'MG', 'IMMP', 'PLCE', 'TDOC', 'INTU', 'NICE', 'EQC', 'UPS', 'VTLE', 'IQV', 'PACB', 'EIX', 'UNF', 'RRX', 'R', 'CHCT', 'MASS', 'GRPN', 'TRMB', 'SMG', 'WSM', 'ESI', 'TIGR', 'HP', 'KNSL', 'ACLS', 'EQNR', 'SFNC', 'WLK', 'SNV', 'AAOI', 'TK', 'ALC', 'UFPI', 'DHT', 'EDN', 'NOC', 'TILE', 'TPH', 'ZLAB', 'CAN', 'FCNCA', 'LXRX', 'BAH', 'FBK', 'DAN', 'PNNT', 'BAC', 'GD', 'PSX', 'CMPR', 'CYD', 'PLAB', 'WNC', 'EVR', 'YRD', 'VTR', 'AXSM', 'MHK', 'SPOT', 'RMD', 'VIPS', 'IOSP', 'FLGT', 'TFC', 'LYV', 'OGS', 'DPZ', 'WAT', 'GTES', 'ZUO', 'AGM', 'TSLX', 'NWBI', 'NSC', 'KRYS', 'PPG', 'NYCB', 'HL', 'CHEF', 'FROG', 'MEI', 'PDM', 'KWR', 'ESE', 'BMA', 'STAG', 'FIBK', 'ENVA', 'GOLF', 'WING', 'APLE', 'YALA', 'SIGI', 'CVX', 'NVEE', 'ASAN', 'AN', 'INVA', 'HTGC', 'CUBI', 'ST', 'AA', 'STRA', 'DE', 'SSL', 'TALO', 'ITGR', 'RY', 'FULT', 'UFI', 'MAXN', 'SYBT', 'TRGP', 'DD', 'PIPR', 'BX', 'BAP', 'COTY', 'EXP', 'UVE', 'CLLS', 'GERN', 'OSIS', 'SAND', 'ADT', 'FLR', 'RJF', 'GDOT', 'OLP', 'BSBR', 'HMST', 'WASH', 'JPM', 'GB', 'MPW', 'PETQ', 'NX', 'EQR', 'EPAC', 'CB', 'EXEL', 'CG', 'MANU', 'GRMN', 'KKR', 'ADSK', 'VST', 'CNI', 'SABR', 'BOH', 'NHI', 'TEVA', 'WY', 'BHF', 'IBN', 'ALGN', 'DAKT', 'GME', 'SPSC', 'BKU', 'REI', 'HPP', 'FITB', 'TMHC', 'CTLT', 'HOPE', 'KMPR', 'BHE', 'CERT', 'AVNT', 'HIW', 'EBS', 'NUS', 'GORO', 'NTR', 'ENB', 'AMRN', 'EXAS', 'FMS', 'MSI', 'HLIT', 'KMX', 'OIS', 'PNR', 'LEGN', 'WK', 'ITCI', 'BNTX', 'BKD', 'CDXS', 'SVM', 'ANIK', 'ACCO', 'ACIW', 'EPR', 'EXC', 'NSA', 'LYFT', 'VLO', 'MAT', 'RYN', 'CIGI', 'EXTR', 'FLEX', 'BALL', 'KTOS', 'ZTS', 'NWS', 'SMP', 'ABEV', 'VEEV', 'DLTR', 'CRM', 'ASUR', 'BANR', 'COLM', 'IRTC', 'ATRI', 'CCS', 'BBDC', 'FBRX', 'ETD', 'SLF', 'CACI', 'TNK', 'HOFT', 'GL', 'TRP', 'ACM', 'TAK', 'E', 'PAGP', 'CFG', 'CTRA', 'BCRX', 'FCX', 'EIG', 'NVAX', 'GOOS', 'CBSH', 'INDB', 'LDOS', 'STLD', 'SITE', 'DCI', 'ESLT', 'OTIS', 'FIVN', 'OM', 'CTVA', 'MFIC', 'KFY', 'VRTX', 'ADUS', 'ARCO', 'MKSI', 'LUMO', 'HMY', 'MRVI', 'ANGI', 'HBI', 'LGIH', 'ZM', 'RSG', 'GDRX', 'MKC', 'WIX', 'IIIN', 'TWI', 'BJ', 'SLGN', 'MASI', 'KRC', 'PGEN', 'EOLS', 'PZZA', 'PJT', 'RRGB', 'ILMN', 'CRI', 'BEAM', 'KNOP', 'DK', 'RDN', 'GIII', 'HALO', 'MGNX', 'PNTG', 'HOLI', 'QTWO', 'PB', 'PEGA', 'EGRX', 'FVRR', 'ENTG', 'CYBR', 'MUR', 'CLVT', 'TGS', 'NMRK', 'BTU', 'ERJ', 'DUK', 'SBS', 'PRLD', 'FRPT', 'FLS', 'EQH', 'KRON', 'PKX', 'MTH', 'SKY', 'LOB', 'BMO', 'JWN', 'ASH', 'CYRX', 'CHRW', 'APPF', 'BOC', 'NWLI', 'JBGS', 'UPWK', 'SUZ', 'SY', 'SMCI', 'MFC', 'CMA', 'BLDR', 'NBIX', 'EBF', 'SCHL', 'TEX', 'MFG', 'CSTE', 'OII', 'BK', 'TNC', 'CCO', 'INSP', 'PRFT', 'CCRD', 'SLRC', 'RES', 'FUBO', 'BRKL', 'ICUI', 'NWSA', 'BILI', 'OLLI', 'VALE', 'NEE', 'REXR', 'GLW', 'NTGR', 'ONTO', 'CRNT', 'EXPE', 'NEM', 'KEN', 'AIZ', 'LCII', 'THO', 'HZO', 'ASR', 'ASML', 'NMFC', 'VVI', 'WSBC', 'H', 'SIM', 'CONN', 'CBRL', 'OFG', 'VLRS', 'YTRA', 'NJR', 'POOL', 'VMI', 'LW', 'CVBF', 'CENX', 'ENSG', 'UNFI', 'KT', 'AOS', 'LC', 'DQ', 'HUBB', 'ALE', 'TROW', 'NXST', 'HLI', 'DLNG', 'RNG', 'EVER', 'PARA', 'TOWN', 'PEP', 'EBAY', 'AME', 'ALNY', 'PNW', 'CLDX', 'BBDO', 'BOOM', 'TAP', 'SAH', 'QRTEB', 'HLIO', 'DOMO', 'LMND', 'HA', 'SYF', 'OXY', 'IT', 'GPK', 'WST', 'WDFC', 'PRAA', 'BBD', 'HD', 'JKHY', 'CAAP', 'FOXA', 'AYI', 'NEOG', 'SD', 'APOG', 'LPSN', 'HTLD', 'CRSR', 'WW', 'CHD', 'AM', 'PYPL', 'MCRB', 'ASIX', 'INFY', 'NHTC', 'TSM', 'RBC', 'BCE', 'PRLB', 'RCKT', 'HESM', 'SAFT', 'RL', 'BHLB', 'KRG', 'REPL', 'ACRS', 'CC', 'STNG', 'STZ', 'HLX', 'IAG', 'TRV', 'GLPI', 'MPAA', 'PVH', 'IRT', 'TCBI', 'PLXS', 'DB', 'FNV', 'ATGE', 'ELF', 'RGLD', 'CUBE', 'SCHW', 'OMC', 'EWBC', 'HOMB', 'SHW', 'UAL', 'FNKO', 'MCO', 'CCK', 'SLM', 'CMCSA', 'CRK', 'POR', 'LKFN', 'BVN', 'ENTA', 'TDG', 'ERIE', 'GOLD', 'MDB', 'AVB', 'TXRH', 'CAR', 'GLPG', 'PTVE', 'IART', 'JNJ', 'KODK', 'ALLY', 'ATSG', 'CEPU', 'PRI', 'GGG', 'AXGN', 'SBSI', 'BYND', 'U', 'NKTR', 'LAB', 'FTDR', 'RGA', 'SASR', 'ELS', 'HCAT', 'MGA', 'DGX', 'DAL', 'APPS', 'CWEN', 'HAFC', 'LECO', 'CPLP', 'YUMC', 'CCU', 'ON', 'WBS', 'TTWO', 'RVLV', 'FAF', 'EMR', 'NNI', 'UCTT', 'AIN', 'VRT', 'TTMI', 'CVGW', 'AC', 'NIU', 'CENT', 'NSP', 'WTM', 'FR', 'CPA', 'ASB', 'SOHU', 'AIT', 'MA', 'AGCO', 'DORM', 'CNDT', 'ITOS', 'DAR', 'AQMS', 'CMRE', 'ADPT', 'LNW', 'PCG', 'AGRO', 'CPB', 'SNPS', 'AOSL', 'NTB', 'ELAN', 'MSA', 'AIG', 'FOR', 'IPG', 'TGTX', 'DBI', 'CPF', 'MUFG', 'TWST', 'AEM', 'KNX', 'ARLO', 'L', 'BIGC', 'CTSH', 'BLX', 'HUBS', 'D', 'SCVL', 'GPN', 'TUP', 'BPOP', 'DNOW', 'SBLK', 'CVI', 'AVNS', 'LKQ', 'BRKR', 'SSP', 'FTV', 'NTNX', 'EGO', 'AVA', 'OKE', 'PLTR', 'CNO', 'JKS', 'HCA', 'AXP', 'BLDP', 'BKNG', 'LOCO', 'FANH', 'APTV', 'TREX', 'FIX', 'CMS', 'ANIP', 'PRGS', 'CDNS', 'SDGR', 'ABT', 'SEM', 'LZB', 'BFH', 'EFSC', 'PATK', 'JBHT', 'TSCO', 'DRH', 'TRS', 'DLTH', 'DHI', 'SU', 'SXC', 'EVRG', 'TW', 'MATV', 'SFL', 'TDC', 'GMED', 'ASO', 'LVS', 'IBM', 'PANW', 'MP', 'FBIN', 'MODG', 'IMO', 'ORN', 'SILC', 'MMSI', 'PBYI', 'CROX', 'GEO', 'CTBI', 'ROKU', 'AORT', 'INMD', 'KMT', 'NOK', 'PBI', 'CAKE', 'OPRA', 'AG', 'HOOK', 'RAMP', 'VIRT', 'GPRK', 'FOLD', 'LPRO', 'GNTX', 'OLED', 'CNP', 'CCRN', 'CSGS', 'JAMF', 'CABO', 'PRMW', 'CCOI', 'GOSS', 'REZI', 'ORIC', 'LLY', 'NHC', 'MAX', 'MRCY', 'WDAY', 'MPWR', 'ENS', 'LMAT', 'OGE', 'WCC', 'SKX', 'PFSI', 'USPH', 'FRHC', 'TKR', 'BDN', 'CARR', 'AVTR', 'OMER', 'AX', 'NDSN', 'NNOX', 'IVZ', 'SIGA', 'MELI', 'OPI', 'ZUMZ', 'IHRT', 'WTS', 'Z', 'AMT', 'RGR', 'TU', 'ARGX', 'HON', 'SSYS', 'ASX', 'CARA', 'PAGS', 'ABR', 'CODI', 'BDX', 'AFL', 'CGEN', 'VRNS', 'RPTX', 'AIR', 'EVTC', 'AWR', 'ARAY', 'CSGP', 'FIVE', 'CRL', 'XPEV', 'SAP', 'AMWD', 'IMMR', 'LOW', 'AKAM', 'SYY', 'YMAB', 'VLY', 'SLCA', 'PSA', 'QRVO', 'ARCC', 'PETS', 'SMHI', 'CTAS', 'CRESY', 'DDS', 'AZTA', 'HOLX', 'NTCT', 'QRTEA', 'GSHD', 'EQX', 'MED', 'AMCR', 'BDTX', 'IRM', 'FE', 'GILD', 'ROCK', 'KR', 'YETI', 'CFFN', 'VYGR', 'JCI', 'VC', 'HST', 'VITL', 'KO', 'CUZ', 'FARO', 'CVAC', 'TFSL', 'AMBC', 'PG', 'YY', 'RDWR', 'PKG', 'SFM', 'GGAL', 'PPL', 'GOGL', 'BZUN', 'LXP', 'PLUG', 'RPM', 'ERIC', 'FSLY', 'RNR', 'HCI', 'TLRY', 'AVY', 'PGNY', 'CBT', 'DAVA', 'EPAM', 'BE', 'BGFV', 'SUPV', 'JOE', 'WMK', 'ASGN', 'AWK', 'PGR', 'OPK', 'RVNC', 'MOH', 'UBSI', 'HCC', 'PI', 'ITRI', 'AL', 'SID', 'C', 'BRO', 'KMI', 'AMD', 'ARDX', 'AJG', 'IIIV', 'ITW', 'BMRN', 'AMWL', 'MANH', 'NCLH', 'SONO', 'CWT', 'ADP', 'KRO', 'SCS', 'HAL', 'JOUT', 'BGNE', 'SNBR', 'CCCC', 'DCO', 'ICL', 'SBCF', 'IRWD', 'AY', 'LEG', 'VZ', 'SRPT', 'VGR', 'AQN', 'TJX', 'STBA', 'NWL', 'INGN', 'GMS', 'JHG', 'MUSA', 'NOVA', 'WAB', 'THC', 'MSM', 'PLAY', 'IONS', 'TV', 'MSTR', 'HOUS', 'CLH', 'AOUT', 'REAL', 'PWR', 'MYGN', 'FCN', 'RGEN', 'AZZ', 'SKYW', 'AVT', 'QTRX', 'SYNA', 'HOG', 'DDOG', 'BTAI', 'SYK', 'JD', 'DAO', 'RMAX', 'SGMO', 'ECPG', 'ITT', 'MPC', 'CPS', 'UI', 'UPLD', 'SAM', 'JHX', 'BRX', 'AKBA', 'PENN', 'RYAAY', 'CX', 'BLFS', 'BSX', 'CHKP', 'NAT', 'KOS', 'PGRE', 'GOGO', 'QCOM', 'VMC', 'MPLN', 'VOYA', 'NWN', 'AES', 'TTEC', 'ATI', 'JBLU', 'UGP', 'DNB', 'FHI', 'YELP', 'BLD', 'LPLA', 'V', 'SUPN', 'WTW', 'TBPH', 'CVE', 'X', 'DSGX', 'REYN', 'FF', 'ATRO', 'SBAC', 'GH', 'FICO', 'FMC', 'COLL', 'DBX', 'STC', 'RARE', 'ZYXI', 'KBH', 'ELV', 'MCFT', 'SGH', 'EGAN', 'INGR', 'XRX', 'ANGO', 'LILA', 'PTC', 'FND', 'SRCL', 'HPE', 'CRTO', 'TRUP', 'LXU', 'CODX', 'BAND', 'CUTR', 'VVV', 'JELD', 'BLUE', 'BIG', 'PDFS', 'INSM', 'MATX', 'TDW', 'DVN', 'GDDY', 'KLAC', 'RCEL', 'UE', 'IPAR', 'DIOD', 'BYD', 'TLYS', 'GPRO', 'DESP', 'MMI', 'AMCX', 'TTD', 'AUDC', 'LYB', 'RCL', 'BAX', 'BFS', 'AZEK', 'BCH', 'PPBI', 'NVRO', 'PFS', 'CLW', 'ADC', 'SLB', 'BA', 'CM', 'RPD', 'AMSF', 'BHC', 'WDC', 'ES', 'KIM', 'ZD', 'BSY', 'TFX', 'FSP', 'NOMD', 'VSAT', 'BANC', 'AGI', 'CIB', 'TCOM', 'VICI', 'CPRI', 'HBM', 'QLYS', 'FNF', 'STTK', 'RYTM', 'LAUR', 'STE', 'SSTK', 'SONY', 'ESGR', 'NNDM', 'NKE', 'DRQ', 'FPI', 'BEN', 'COKE', 'CMP', 'DECK', 'VIR', 'VBTX', 'BZH', 'AFYA', 'FIS', 'WABC', 'CLF', 'PRGO', 'FLIC', 'MMS', 'AMG', 'CVS', 'AHH', 'HDB', 'WM', 'CFR', 'ZBRA', 'ORA', 'TGNA', 'FCPT', 'BURL', 'DVAX', 'ALRM', 'XPEL', 'SUI', 'PSMT', 'LAMR', 'UGI', 'BKE', 'AMP', 'GIL', 'NG', 'XHR', 'EVH', 'ADBE', 'OKTA', 'IBP', 'HES', 'UHS', 'DTE', 'ICE', 'ACN', 'MLAB', 'FSK', 'SITM', 'BERY', 'ZS', 'EMN', 'KRNY', 'ABG', 'UFCS', 'PAYC', 'XP', 'IDXX', 'MGPI', 'G', 'LPG', 'CSL', 'ESTC', 'SEB', 'TROX', 'VRSK', 'MEDP', 'WHD', 'MCS', 'DYN', 'CE', 'MT', 'APPN', 'ACI', 'INCY', 'CERS', 'HWM', 'RYAM', 'HEI', 'NVST', 'FOSL', 'ANET', 'ACIU', 'MTN', 'DHIL', 'NFG', 'KTB', 'GTN', 'FGEN', 'AGIO', 'ABCB', 'FAST', 'BLNK', 'FWRD', 'JBSS', 'VCTR', 'MGRC', 'ALEC', 'HTLF', 'ETSY', 'TLS', 'FNB', 'BL', 'FDS', 'VRRM', 'PFBC', 'COF', 'LRCX', 'NEXA', 'CACC', 'CNOB', 'SPOK', 'AMN', 'PAYS', 'POST', 'KC', 'PPC', 'TXT', 'VNO', 'LPX', 'GES', 'WSO', 'AMKR', 'KBR', 'LEA', 'MGM', 'INN', 'GPS', 'KALU', 'BABA', 'AFG', 'BRC', 'BLK', 'SLAB', 'JJSF', 'LE', 'VCEL', 'ATUS', 'CINF', 'SRG', 'OMAB', 'BPMC', 'GLNG', 'EZPW', 'ERII', 'EQT', 'GS', 'DRD', 'EHTH', 'ECL', 'FN', 'ATO', 'CSWI', 'BMI', 'SFBS', 'OC', 'MCK', 'PUMP', 'FIZZ', 'TXMD', 'QXO', 'LWLG', 'CSLR', 'FLNC', 'PTLO', 'GE', 'GOCO', 'COUR', 'DSGN', 'FIGS', 'RR', 'TBRG', 'SPCE', 'ATAI', 'LAAC', 'ZEPP', 'ACUT', 'RBOT', 'SHECY', 'BTCM', 'AEON', 'LOGC', 'OABI', 'DBRG', 'AGFY', 'BNZI', 'BMBL', 'HDL', 'UBX', 'SPNT', 'LAZ', 'CGEM', 'QTTB', 'GEV', 'APO', 'ZWS', 'BN', 'INTA', 'SQSP', 'PRE', 'ABEO', 'YMM', 'DNTH', 'DUFRY', 'AMPL', 'TIL', 'RETO', 'CSWC', 'LLYVK', 'LENZ', 'ALUR', 'LFST', 'BRNS', 'WOLF', 'HY', 'ZURVY', 'TORO', 'DLO', 'ATNF', 'CURV', 'TALK', 'NPSNY', 'SMRT', 'EHAB', 'ONL', 'ENVX', 'KNDI', 'HOOD', 'EDR', 'BLZE', 'ZK', 'PGRU', 'LDTC', 'COCH', 'NCNO', 'BG', 'INKT', 'ULCC', 'PIII', 'MARA', 'QFIN', 'NATL', 'CXT', 'HG', 'CGNT', 'SMBK', 'ALTM', 'LIF', 'CSAN', 'OCFT', 'RGS', 'DTIL', 'BEST', 'IOBT', 'NE', 'ALKT', 'PCOR', 'CLPBY', 'PSFE', 'PROC', 'LANV', 'GLT', 'MRDB', 'NRDY', 'BROS', 'GXO', 'ALHC', 'NCNA', 'J', 'FG', 'CMPO', 'CTRM', 'VSCO', 'PHUN', 'XBP', 'SLDB', 'CREV', 'ARM', 'RIGL', 'BQ', 'FLGC', 'OGI', 'ZH', 'NRXP', 'MESO', 'AIRS', 'PAL', 'SGN', 'LOT', 'WNS', 'TCRT', 'TBLA', 'AUR', 'TCEHY', 'HCP', 'TRST', 'IRAA', 'GROV', 'INO', 'BACHY', 'AZPN', 'IMAX', 'DRS', 'COO', 'FNA', 'CLB', 'TPG', 'KD', 'MOND', 'UAVS', 'PGY', 'LVLU', 'EFXT', 'MBC', 'WFG', 'REE', 'OCX', 'LCID', 'SEEL', 'CADE', 'BZ', 'PR', 'VSTS', 'EXAI', 'ATRA', 'FA', 'CTV', 'ADEA', 'RUM', 'DUOL', 'BEDU', 'INFA', 'VWDRY', 'NSANY', 'PLMI', 'HYPR', 'WBD', 'WRBY', 'WKME', 'TSAT', 'ZGN', 'OBK', 'OP', 'PATH', 'LSXMA', 'RERE', 'FRO', 'IVCGF', 'CISO', 'IBRX', 'SWI', 'CTLP', 'AVDX', 'MQ', 'TBIO', 'CXM', 'DRRX', 'BBAI', 'ZIP', 'BDORY', 'MOGO', 'LICY', 'VZIO', 'KNSA', 'GLBE', 'WGS', 'RCM', 'STHO', 'HTZ', 'IHS', 'MBLY', 'AMBI', 'BNGO', 'CWBC', 'ORGN', 'SEMR', 'SDIG', 'ELPC', 'LSPD', 'CORZ', 'RKLB', 'AEG', 'AILE', 'HAYW', 'CIVI', 'BRBR', 'EMKR', 'RWAY', 'TLPH', 'IBTA', 'GILT', 'INDI', 'CHPT', 'LLYVA', 'SKLZ', 'RIVN', 'MNTS', 'ZIMV', 'SKWD', 'GIC', 'MUX', 'DNZOY', 'S', 'SWIM', 'NKLA', 'NB', 'WBX', 'NABL', 'SEAT', 'DBVT', 'OCGN', 'CNF', 'BOW', 'RBLX', 'BWIN', 'JZ', 'RUSHA', 'HYMC', 'KALA', 'FIP', 'BNAI', 'KIND', 'UHG', 'GCT', 'MMAT', 'PHIN', 'GRNT', 'CRIS', 'PRAX', 'SOS', 'OKLO', 'DOYU', 'SOLV', 'TDUP', 'AMTB', 'QBTS', 'FANUY', 'PROK', 'INBX', 'SPXC', 'ROOT', 'JOBY', 'AQB', 'SLDP', 'BRZE', 'SBGI', 'ASTH', 'ACB', 'SLVM', 'SOFI', 'GETR', 'CCG', 'SVCO', 'MOMO', 'IRS', 'ALNT', 'MTTR', 'FTRE', 'FSM', 'EVTL', 'EMBC', 'CTNM', 'PSNY', 'ACIC', 'SCLX', 'DIDIY', 'EGIO', 'HIMS', 'PRVA', 'LOCL', 'AFMD', 'YSG', 'CRBP', 'FRGE', 'VIGL', 'GFS', 'YI', 'ADTN', 'PACS', 'AMC', 'QGEN', 'IMCC', 'GAUZ', 'TRVG', 'LGO', 'ACT', 'LYEL', 'WDH', 'THCH', 'CLS', 'CYTH', 'VTS', 'ALPP', 'DJT', 'EVGO', 'BODI', 'OPAL', 'BTE', 'STKH', 'NLY', 'NRDS', 'ONON', 'RXO', 'FTFT', 'CP', 'SLN', 'FRSH', 'CPAY', 'OMIC', 'VRN', 'TECX', 'EDU', 'FLYX', 'FRCOY', 'GTLB', 'MTEM', 'BWMX', 'SYRS', 'SGHC', 'DTC', 'CIXXF', 'APP', 'ACVA', 'OUST', 'WTTR', 'ATIP', 'BEEP', 'SLG', 'IE', 'OPFI', 'VIV', 'VIOT', 'BATRK', 'RNW', 'SNCR', 'ALVO', 'MBRX', 'DOMA', 'TFPM', 'CRDO', 'SKYX', 'AXDX', 'HR', 'MULN', 'MDAI', 'STEL', 'SOC', 'ONCT', 'AVDL', 'CLBT', 'FBYD', 'BKKT', 'EVCM', 'SPRU', 'GETY', 'DOCN', 'KPLT', 'AMX', 'VLD', 'DSGR', 'TROO', 'BLDE', 'DNUT', 'JXN', 'ME', 'HYZN', 'ECX', 'CEG', 'PAYO', 'WTO', 'TOON', 'BNED', 'RDDT', 'MVST', 'ZKH', 'FREY', 'TASK', 'UP', 'ANTE', 'UXIN', 'AVAH', 'SMLR', 'GGR', 'CPNG', 'CXAI', 'LEV', 'FRPH', 'MNY', 'LSXMK', 'FBMS', 'CCCS', 'HEPS', 'AMBP', 'CHAA', 'DDL', 'LAC', 'FLYW', 'COMP', 'EVEX', 'FLNT', 'WS', 'PL', 'ENFN', 'RGTI', 'SATL', 'HHH', 'BARK', 'CSSE', 'BZFD', 'PRCT', 'AFCG', 'ENOV', 'APA', 'BHVN', 'GBTG', 'NN', 'NLOP', 'DKNG', 'TLIS', 'MURA', 'AVPT', 'TBN', 'WAY', 'IDEX', 'COHR', 'CRGX', 'KLG', 'AISP', 'SES', 'VCSA', 'WBTN', 'SOND', 'DINO', 'GEHC', 'CR', 'LZ', 'BAER', 'MRVL', 'OSCR', 'AMTD', 'FENG', 'TOI', 'SMSI', 'BHM', 'PYXS', 'DNA', 'CELU', 'BBUC', 'XOS', 'NVTS', 'CFLT', 'AKLI', 'LTH', 'DM', 'BFLY', 'ONMD', 'PERF', 'DAVE', 'MNMD', 'PRKS', 'FFIE', 'CINT', 'XPER', 'BOLD', 'OLPX', 'NERV', 'LIDR', 'TFFP', 'AMLX', 'HNST', 'TWOU', 'ATER', 'ARIS', 'VLTO', 'SLNA', 'WNW', 'GRAB', 'ABL', 'LFCR', 'AIP', 'ALTI', 'NU', 'ALMS', 'OTRK', 'OTLY', 'BNR', 'LH', 'IOT', 'PDYN', 'ULS', 'NCTY', 'AMPS', 'COIN', 'DV', 'KORE', 'CANG', 'FWONK', 'WATT', 'DOUG', 'BGC', 'JBI', 'BBWI', 'CCSI', 'CMAX', 'ARQQ', 'LUNR', 'DKILY', 'INTZ', 'DDC', 'LIN', 'BRCC', 'LB', 'CTRI', 'EBON', 'HLMN', 'LVWR', 'ASAI', 'RVYL', 'ECVT', 'BNRE', 'RENB', 'TWKS', 'OGN', 'HROW', 'FTCI', 'SGHT', 'TMC', 'WALD', 'ODP', 'TLSI', 'BETR', 'BATRA', 'ALIT', 'LILM', 'METCB', 'EQBK', 'HLLY', 'STR', 'TRDA', 'NSRGY', 'VRM', 'SHCO', 'RENT', 'EXFY', 'SHCR', 'MNDY', 'TOST', 'GOEV', 'SNDL', 'KNF', 'RAPP', 'XMTR', 'LBTYK', 'KRRO', 'IONQ', 'LZAGY', 'TRML', 'SKIL', 'FWONA', 'ARHS', 'LIFW', 'BKSY', 'EXTO', 'VSTM', 'ROIV', 'ESAB', 'CVII', 'ICLK', 'LUCD', 'SWVL', 'CDRE', 'LLAP', 'DTM', 'TCBX', 'LOVE', 'MKTW', 'GTBP', 'CGTX', 'RBA', 'RUSHB', 'STG', 'SMWB', 'NGNE', 'FCFS', 'RCRUY', 'VBIV', 'MCW', 'XIN', 'INZY', 'ERAS', 'BAM', 'NEXI', 'TISI', 'AMKBY', 'OWLT', 'SHLS', 'NRGV', 'UDMY', 'ASRT', 'WEAV', 'SOGP', 'MSGE', 'GORV', 'DFH', 'PAVM', 'PLL', 'WPRT', 'VTYX', 'RDHL', 'TKO', 'CGC', 'ELP', 'FRSX', 'IAC', 'SYRE', 'NAPA', 'NEUE', 'TNL', 'GLAD', 'DRTS', 'BHIL', 'GCBC', 'MSGM', 'ADN', 'DUO', 'ML', 'EVA', 'SRAD', 'AEVA', 'GRND', 'OCSL', 'FTAI', 'VMEO', 'RGF', 'XLO', 'IAS', 'TSVT', 'TSE', 'DH', 'CMCM', 'MIR', 'GWH', 'ELTX', 'CRCT', 'LSAK', 'ZETA', 'SG', 'ACHR', 'SKM', 'AU', 'OPAD', 'NTDOY', 'CFRUY', 'FMEd', 'GMET', 'HERC', 'CTA', 'ETP', 'NMT', 'WISE', 'VRCI', 'REVB', 'PULS', 'SHEL', 'GSK', 'NWG', 'BHP', 'INDV', 'HLN', 'WPS', 'WDS', 'PINE', 'RGT', 'EARN', 'TIDE', 'TRX', 'ASTO', 'SNT', 'CDL', 'CGO', 'XAR', 'DEC', 'SORT', 'EBET', 'COp', 'ACp', 'SDG', 'PET', 'ICG', 'FNX', 'CLCO', 'EXR', 'SRE', 'BME', 'PFD', 'VOD', 'FDM', 'ARCM', 'ZIG', 'ESP', 'ATYM', 'BAB', 'BUR', 'KIE', 'EQLS', 'NEXN', 'EZJ', 'PRM', 'ARB', 'EVOK', 'DIAL', 'RACE', 'TCX', 'ISRG', 'VRTS', 'FANG', 'ALTO', 'AMRX', 'CNK', 'WU', 'PFX', 'AAPL', 'NPO', 'MKL', 'ALGM', 'SNAP', 'F', 'MSFT', 'VTRS', 'ATNI', 'CNH', 'NVDA', 'LAZR', 'XOM', 'CNXC', 'ELME', 'LTC', 'HUBG', 'TECH', 'COR', 'GGB', 'GOOGL', 'KLXE', 'K', 'NOV', 'WBA', 'CENTA', 'DRI', 'NYT', 'NIO', 'CMI', 'CIG', 'DHR', 'BWA', 'DRVN', 'CBU', 'WMT', 'NVO', 'GSBD', 'FET', 'CMC', 'PRG', 'CELH', 'MTD', 'CBRE', 'RS', 'KSS', 'RNST', 'PRCH', 'WOR', 'MODV', 'HSY', 'HNI', 'AIOT', 'LEVI', 'SSB', 'BBSI', 'CI', 'FSRNQ', 'AAN', 'SIG', 'EURN', 'PLTK', 'FFWM', 'BXP', 'EFC', 'AI', 'CCI', 'LPL', 'DEA', 'PAAS', 'SBUX', 'META', 'IP', 'RDUS', 'TRUE', 'ODFL', 'FLWS', 'DASH', 'EVRI', 'DOC', 'RIOT', 'SSTI', 'WOOF', 'SOL', 'KAR', 'ARQ', 'MNST', 'NVRI', 'ROK', 'GM', 'UNIT', 'IRBT', 'EXLS', 'MSGS', 'NTES', 'HASI', 'EXPD', 'SAIC', 'UHAL', 'KFRC', 'SNDA', 'HMC', 'CLOV', 'TSLA', 'ONIT', 'TER', 'AFRM', 'CHDN', 'OEC', 'ABNB', 'PUBM', 'BSIG', 'GEVO', 'TKC', 'AAON', 'STWD', 'MAR', 'TFIN', 'XPO', 'IIPR', 'RITM', 'RTX', 'SPNS', 'OPEN', 'EQIX', 'CNQ', 'SPHR', 'AMZN', 'WPC', 'VYX', 'OBDC', 'CSTL', 'DAY', 'CMPS', 'PCAR', 'KHC', 'WKC', 'LNC', 'CHT', 'PDD', 'SNEX', 'UPBD', 'NFLX', 'BYON', 'EB', 'NSIT', 'BUD', 'TM', 'FL', 'EOSE', 'NNN', 'TVTX', 'EG', 'MTCH', 'RVTY', 'TPL', 'STLA', 'CMG', 'MLI', 'O', 'ROP', 'MDLZ', 'CHRS', 'EL', 'CDP', 'MTUS', 'EH', 'TR', 'CPRT', 'NDAQ', 'ALKS', 'CGNX', 'WT', 'LRN', 'SAVE', 'NWE', 'LESL', 'NMRA', 'LOAR', 'JBSAY', 'CAVA', 'CART', 'BLTE', 'BLRX', 'SPIR', 'RBRK', 'ALAB', 'SGML', 'HUT', 'MTEK', 'GSIT', 'LIPO', 'GOOG', 'GUTS', 'PLYM', 'SDA', 'GROM', 'OTLK', 'HSAI', 'SDHC', 'SMR', 'LNZA', 'DO', 'SPRC', 'CWCO', 'MXCT', 'BITF', 'TNON', 'AMPX', 'BTSG', 'ODD', 'ISDR', 'CRBG', 'SCYX', 'TNGX', 'ANL', 'VFS', 'EU', 'UROY', 'DXLG', 'FEMY', 'CDIO', 'SND', 'GBOOY', 'KYTX', 'TANH', 'BIRK', 'DALN', 'IMPUY', 'NUVL', 'URG', 'PHIO', 'APLD', 'MOB', 'JAKK', 'DOLE', 'HNNA', 'NEXT', 'AS', 'JCTCF', 'TZOO', 'LXEO', 'FLUX', 'ABAT', 'SAMG', 'CYBN', 'RSKD', 'INVO', 'TTI', 'DCGO', 'ICU', 'DDI', 'EWCZ', 'SLRN', 'NAII', 'VRDN', 'GLDG', 'SHIM', 'PANL', 'SHOT', 'LMB', 'THTX', 'SBR', 'ABVC', 'SCPH', 'PLX', 'USGO', 'CIFR', 'RC', 'TAYD', 'CGON', 'ANRO', 'SOUN', 'BNOX', 'MRX', 'FNGR', 'WULF', 'WFRD', 'AHR', 'NTRP', 'ONCY', 'AESI', 'SURG', 'TPST', 'ESOA', 'BORR', 'KVYO', 'NPWR', 'MLTX', 'AUNA', 'TBBB', 'VIK', 'AVBP', 'NLST', 'IMMX', 'Gm', 'ACXP', 'EE', 'KVUE', 'AEHR', 'PRZO', 'MGX', 'VTSI', 'WTI', 'PRME', 'MYO', 'TRMD', 'IAUX', 'GNE', 'SVV', 'NCMI', 'IREN', 'HIVE', 'UG', 'IMRX', 'RVPH', 'NVOS', 'SFLm', 'CEm', 'IGm', 'TBLD', 'ESp', 'MDMp', 'TFFp', 'MLp', 'AMp', 'TRIp', 'DECp', 'BBp', 'LBTYA', 'SANp', 'DGp', 'ELp', 'SUp', 'AIp']

print(len(tickers))


3944


In [11]:
import pandas as pd
import yfinance as yf
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns

# List of 100 stock tickers
#tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'FB', 'NVDA', 'NFLX', 'BABA', 'V']

# Download historical data for the stocks
data = yf.download(tickers, start="2020-01-01", end="2023-01-01")['Adj Close']

# Calculate historical returns
returns = data.pct_change().mean() * 252

# Sort tickers based on returns
sorted_tickers = returns.sort_values(ascending=False).index.tolist()

# Function to group tickers into sets of 10
def group_tickers(tickers, group_size=10):
    return [tickers[i:i + group_size] for i in range(0, len(tickers), group_size)]

# Group the sorted tickers
grouped_tickers = group_tickers(sorted_tickers)

# Function to fetch stock data and calculate portfolio metrics
def analyze_portfolio(tickers):
    try:
        data = yf.download(tickers, start="2020-01-01", end="2023-01-01")
        prices = data['Adj Close']
        volumes = data['Volume']

        # Calculate expected returns and sample covariance matrix
        mu = expected_returns.mean_historical_return(prices)
        S = risk_models.sample_cov(prices)

        # Optimize for maximum Sharpe ratio
        ef = EfficientFrontier(mu, S)
        weights = ef.max_sharpe()
        cleaned_weights = ef.clean_weights()

        # Convert weights to percentages
        percentage_weights = {ticker: weight * 100 for ticker, weight in cleaned_weights.items()}

        # Calculate the expected annual return, annual volatility, and Sharpe ratio
        performance = ef.portfolio_performance(verbose=True)

        # Fetch additional stock data
        stock_info = yf.Tickers(tickers)

        stock_data = []
        for ticker in tickers:
            try:
                info = stock_info.tickers[ticker].info
                stock_data.append({
                    'Ticker': ticker,
                    'Price': prices[ticker].iloc[-1],
                    'Weight (%)': round(percentage_weights[ticker], 2),  # Display weight as percentage
                    'Volume': volumes[ticker].iloc[-1],
                    'Previous Volume': volumes[ticker].iloc[-2],
                    'All-Time High': info['fiftyTwoWeekHigh'],
                    '52-Week High': info['fiftyTwoWeekHigh'],
                    'Target': info['targetMeanPrice'] if 'targetMeanPrice' in info else None
                })
            except Exception as e:
                print(f"Skipping invalid/unavailable ticker {ticker}: {e}")
        return stock_data, performance

    except ValueError as e:
        print(f"Error processing tickers: {tickers}. Moving on... Error: {e}")
        return None, None

# Analyze each group and display the results in the notebook
for i, group in enumerate(grouped_tickers):
    stock_data, performance = analyze_portfolio(group)
    if stock_data:
        df = pd.DataFrame(stock_data)
        print(f"\nSet {i + 1} Portfolio Performance:")
        print(f"Expected annual return: {performance[0]:.2%}")
        print(f"Annual volatility: {performance[1]:.2%}")
        print(f"Sharpe Ratio: {performance[2]:.2f}")
        print("\nStock Data:")
        display(df)


[*********************100%***********************]  3933 of 3933 completed

139 Failed downloads:
['HG', 'CREV', 'ALMS', 'CLCO', 'BQ', 'EQLS', 'NATL', 'INOV', 'KYTX', 'AVBP', 'ANL', 'PAL', 'PRZO', 'MRX', 'FUSI', 'LRE', 'GUTS', 'ULS', 'SGN', 'SVV', 'ECO', 'BOLD', 'RDDT', 'CAVA', 'NB', 'LLYVK', 'MGX', 'CTNM', 'BCG', 'AESI', 'VSTS', 'BIRK', 'LXEO', 'CAML', 'ANRO', 'MURA', 'TBBB', 'ELPC', 'SDHC', 'INHD', 'CRW', 'QSG', 'BEEP', 'FBYD', 'VTS', 'CGON', 'EXTO', 'LOAR', 'RR', 'KVYO', 'CRGX', 'LPA', 'CORZ', 'ZKH', 'MLPD', 'AEON', 'TBN', 'PTEC', 'TEM', 'SHIM', 'NXT', 'GAUZ', 'SGE', 'LIF', 'DDC', 'CCG', 'PACS', 'FTRE', 'AHR', 'HDL', 'BTSG', 'KNF', 'FLYX', 'NMRA', 'FMED', 'IBTA', 'LLYVA', 'LAC', 'VIK', 'AS', 'VLTO', 'CARD', 'ICG', 'CSLR', 'CR', 'INBX', 'WS', 'ISPY', 'MSGE', 'TORO', 'DECP', 'BOW', 'ZK', 'SOLV', 'GEV', 'HSAI', 'RBRK', 'USGO', 'PSH', 'SHLD', 'ARBB', 'WAY', 'ALAB', 'RAPP', 'LB', 'WISE', 'KLG', 'STHO', 'SKWD', 'SLRN', 'ODD', 'DEC', 'VFS', 'EVSB', 'KVUE', 'CTRI', 'NLOP', 'AUNA', 'METCB', 

ERROR in LDL_factor: Error in KKT matrix LDL factorization when computing the nonzero elements. The problem seems to be non-convex
ERROR in osqp_setup: KKT matrix factorization.
The problem seems to be non-convex.


SolverError: Workspace allocation error!